# 🧪 LAB: Clustering several real datasets

**Motivation**: This lab adopts three contrasting datasets to illustrate how clustering behaves under different real-world conditions—differences in scale, varying cluster separations, and high-dimensional gene expression data. You will put these concepts into practice by applying different clustering methods and evaluating how well they uncover structure in each dataset.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


NAME(s) = ""

## Part 1: Clustering arrests by U.S. state (25 points)

The dataset below contains statistics for the 50 U.S. states on four variables recorded in 1973:

- Murder: Arrests for murder (per 100,000 residents)
- Assault: Arrests for assault (per 100,000 residents)
- UrbanPop: Percentage of urban population
- Rape: Arrests for rape (per 100,000 residents)

1. Perform hierarchical clustering using complete linkage and Euclidean distance. Plot the resulting dendrogram.

2. Cut the dendrogram at a height that produces three clusters. Which states belong to each cluster?

3. Repeat the hierarchical clustering, again using complete linkage and Euclidean distance, after scaling all variables to have a standard deviation equal to one.

4. Compare the two clustering solutions. What effect does scaling the variables have on the resulting dendrogram and cluster memberships?
In your opinion, should the variables be scaled in this case before computing inter-observation dissimilarities? Provide a justification for your answer.

In [ ]:
import pandas as pd 
dat_1 = pd.read_csv("Lab08-part1_data.csv")
states = dat_1.state.to_list()
X = dat_1.drop(columns="state")
X.head() # This is the dataset to use

In [ ]:
# USE AS MANY CELLS AS NEEDED

## Part 2: Clustering, genomic data, and cancer (35 points)

Unsupervised techniques are often used to analyze genomic data. In particular, PCA and hierarchical clustering are popular tools.
You will apply these techniques to the dataset below, which consists of gene expression measurements for 64 cancer cell lines across 6,830 genes.

1. Perform hierarchical clustering of the data using complete, single, and average linkage. Use Euclidean distance as the distance metric in all three cases.

2. Plot the dendrogram for each linkage method. Briefly describe the structure of these dendrograms and comment on the main differences produced by the three linkage criteria.

3. Using the known cancer type labels and examining each dendrogram, qualitatively describe how well the samples cluster into distinct cancer types. Do you see clear separation between some types? Which ones appear more mixed?

4. Focusing on complete linkage, plot the dendrogram again and cut it to obtain 4 clusters (e.g., highlight or color the branches). Briefly describe which cancer cell lines cluster together (for example, which cancer types tend to appear in the same cluster). Compute internal validation scores for this 4-cluster partition (e.g., Silhouette score, Calinski–Harabasz index, Davies–Bouldin index).

5. Fit a K-Means model with 4 clusters to the same data. Compare this 4-cluster K-Means solution with the 4-cluster hierarchical solution from step 4. How similar are the two partitions? Use at least one external validation metric (e.g., Adjusted Rand Index) to quantify their agreement.

6. Genomic data can be noisy and high-dimensional. To address this, apply PCA to the original data and reduce it to the first 5 principal components. Then perform hierarchical clustering with complete linkage and Euclidean distance on this reduced 5-dimensional representation. Cut the resulting dendrogram to obtain 4 clusters.

7. Compute the internal validation metrics for this PCA-based 4-cluster solution. Do these values indicate better or worse quality than those obtained using the original (non-reduced) data? In addition, compare the agreement between the PCA-based partition and the original partition using at least one external validation metric. Comment on whether dimensionality reduction improved, worsened, or did not change the clustering quality.

In [ ]:
import pandas as pd
X = pd.read_csv("Lab08-part2_data.csv") # This is the dataset to use
# These are the labels for each observation
y = pd.read_csv("Lab08-part2_labels.csv").label.to_numpy()

In [ ]:
# USE AS MANY CELLS AS NEEDED

## Part 3: Clustering gene expression between healthy and diseased groups (35 points)

The following dataset contains gene expression measurements for 40 tissue samples, each measured on 1,000 genes. The first 20 samples come from healthy patients, while the second 20 samples come from a diseased group.

1. Apply hierarchical clustering to the samples using a correlation-based distance. Use complete, average, and single linkage. Plot the resulting dendrogram for each case.

2. Looking at these dendrograms, which linkage criterion seems to provide better separation of the two groups?

3. Why do you think you were not asked to use Ward linkage here?

4. Now, focusing on complete linkage, cut the dendrogram into 2 clusters. Compare this clustering with the known group labels using the Adjusted Rand Index.

5. Use a two-sample t-test (e.g., the `ttest_ind` function from `scipy.stats`) to compute a p-value for each gene assessing its ability to discriminate between the two groups. There are 1000 genes, so apply a Bonferroni correction by using a significance threshold of α/1000 (with α = 0.05). In other words, consider a gene statistically significant if its p-value is less than 0.05/1000. Select all genes that meet this criterion.

6. Reapply hierarchical clustering to the samples, again with complete linkage and correlation-based distance, but now using only the significant genes. Cut the tree into 2 clusters and compute the Adjusted Rand Index with respect to the known labels. Which clustering solution shows better agreement with the true groups?

7. In the previous step, you should observe that clustering using only the significant genes leads to worse agreement with the known labels than using all 1000 genes, even though these genes were selected for their discriminative ability. Now repeat the hierarchical clustering using complete linkage and Euclidean distance on the significant genes. Cut the dendrogram into 2 clusters and compare the results with the known labels. Why does Euclidean distance produce a much better reconstruction than correlation distance in this scenario?

In [ ]:
import pandas as pd
import numpy as np

X = pd.read_csv("Lab08-part3_data.csv") #This is the dataset to use
#We know that the first 20 observations correspond to healthy patients, and the remaining 20 to disease. Let's create labels then:
y = np.array(["healthy"]*20 + ["disease"]*20)

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import ttest_ind
from sklearn.metrics import adjusted_rand_score

print(X.shape)  # 40 samples x 1,000 genes

### 1. Hierarchical clustering with correlation-based distance

In [ ]:
#Correlation distance between samples: 1 - Pearson correlation of their gene expression profiles
D_corr = pdist(X, metric="correlation")

Z3 = {}
fig, axes = plt.subplots(3, 1, figsize=(16, 15))
for ax, method in zip(axes, ["complete", "average", "single"]):
    Z3[method] = linkage(D_corr, method=method)
    dendrogram(Z3[method], labels=y, leaf_rotation=90, ax=ax)
    ax.set_title(f"{method.capitalize()} Linkage, Correlation Distance")
    ax.set_ylabel("Height")
plt.tight_layout()
plt.show()

In [ ]:
#How well a 2-cluster cut of each tree matches the true groups
for method, Z in Z3.items():
    c = fcluster(Z, t=2, criterion="maxclust")
    print(f"{method:>8}: ARI = {adjusted_rand_score(y, c):.3f}")
    print(pd.crosstab(y, c, rownames=["Group"], colnames=["Cluster"]), "\n")

### 2. Which linkage separates the groups best?

Complete linkage works best. In its dendrogram, all 20 diseased samples are on one side and all 20 healthy samples are on the other. The diseased samples are very similar to each other, so they join together low in the tree. The healthy samples are less similar, so they join higher up. Average linkage also keeps the diseased samples together, but healthy samples end up on both sides of them. Single linkage adds samples one at a time ("chaining"), so it shows no clear two groups.

### 3. Why not Ward linkage?

Ward linkage only works with Euclidean distance. It joins the clusters that add the least variance, and it measures that variance using Euclidean distance to each cluster's center. We are using a correlation-based distance here, so Ward's results wouldn't make sense.

### 4. Complete linkage cut into 2 clusters

In [ ]:
clusters_corr = fcluster(Z3["complete"], t=2, criterion="maxclust")
print(pd.crosstab(y, clusters_corr, rownames=["Group"], colnames=["Cluster"]))
print("\nAdjusted Rand Index:", round(adjusted_rand_score(y, clusters_corr), 3))

All 20 diseased samples end up in one cluster, but the healthy samples are split in half: 10 join the diseased cluster and 10 form their own. The Adjusted Rand Index is about 0.24, so the clusters only weakly match the true groups. The healthy samples aren't very similar to each other, so the tree's top split divides the healthy group instead of separating healthy from diseased.

### 5. t-test for each gene with Bonferroni correction

In [ ]:
#Two-sample t-test for every gene (column): healthy (first 20 rows) vs. disease (last 20 rows)
t_stat, p_values = ttest_ind(X.iloc[:20], X.iloc[20:], axis=0)

#Bonferroni correction: significant if p < 0.05 / 1000
alpha = 0.05 / X.shape[1]
sig_genes = X.columns[p_values < alpha]
print(f"Bonferroni threshold: {alpha}")
print(f"Number of significant genes: {len(sig_genes)}")
print("Significant genes:", list(sig_genes))

### 6. Re-cluster using only the significant genes (correlation distance)

In [ ]:
X_sig = X[sig_genes]
Z_sig_corr = linkage(pdist(X_sig, metric="correlation"), method="complete")

plt.figure(figsize=(16, 5))
dendrogram(Z_sig_corr, labels=y, leaf_rotation=90)
plt.title("Complete Linkage, Correlation Distance, Significant Genes Only")
plt.ylabel("Height")
plt.show()

clusters_sig_corr = fcluster(Z_sig_corr, t=2, criterion="maxclust")
print(pd.crosstab(y, clusters_sig_corr, rownames=["Group"], colnames=["Cluster"]))
print("\nAdjusted Rand Index:", round(adjusted_rand_score(y, clusters_sig_corr), 3))

Using only the significant genes, the clusters are basically random: each cluster has a mix of healthy and diseased samples, and the ARI is about **0.00**. Clustering with all 1,000 genes (ARI ≈ 0.24) matches the true groups better.

### 7. Re-cluster the significant genes with Euclidean distance

In [ ]:
Z_sig_euc = linkage(X_sig, method="complete", metric="euclidean")

plt.figure(figsize=(16, 5))
dendrogram(Z_sig_euc, labels=y, leaf_rotation=90)
plt.title("Complete Linkage, Euclidean Distance, Significant Genes Only")
plt.ylabel("Height")
plt.show()

clusters_sig_euc = fcluster(Z_sig_euc, t=2, criterion="maxclust")
print(pd.crosstab(y, clusters_sig_euc, rownames=["Group"], colnames=["Cluster"]))
print("\nAdjusted Rand Index:", round(adjusted_rand_score(y, clusters_sig_euc), 3))

In [ ]:
#Average expression of the significant genes in each group
print("Mean expression of significant genes, healthy:", round(X_sig.iloc[:20].values.mean(), 2))
print("Mean expression of significant genes, disease:", round(X_sig.iloc[20:].values.mean(), 2))

With Euclidean distance, the clusters match the true groups perfectly (ARI = 1.0).

Why does Euclidean work better here? The significant genes are simply higher in diseased samples: about 2.0 on average, compared with about 0.0 in healthy samples. Euclidean distance measures how far apart the actual values are, so it easily tells the two groups apart. Correlation distance only looks at the pattern of ups and downs across genes, and it ignores whether values are high or low overall. Since all the significant genes go up together in diseased samples, the pattern looks the same in both groups, and correlation can't tell them apart.

## Part 4: Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

YOUR TEXT HERE